# 1. Project Overview and Goal

## 1.1. Dynamic Crowd Modelling Objectives
*(Content to be added)*

## 1.2. Domain Terminology and Physical Definitions

In order to maintain accuracy, our system adheres to standard physical properties, calculations, and methodologies. The most important definitions are presented below to establish key concepts and ensure the reader is fully aligned with the terminology referenced throughout this work.

### 1.2.1. Sensors
*   **Accelerometer:** Measures proper linear acceleration (\(\text{m/s}^2\)) along three orthogonal axes (\(x, y, z\)), registering user-driven dynamic force combined with static Earth gravity.
*   **Gyroscope:** Measures angular velocity (\(\text{rad/s}\)) about the three axes, tracking the rate of device rotation independent of linear force.

### 1.2.2. Telemetry
*   **Acceleration Magnitude (\(\|a\|\)):** The absolute kinetic intensity of an agent, calculated as the Euclidean norm of the spatial components, \(\|a\| = \sqrt{x^2 + y^2 + z^2}\), measured in metres per second squared (\(\text{m/s}^2\)). It serves as our proxy for active stride frequencies and transit movements.
*   **6D Orientation Vector:** The aligned combination of three-axis accelerometer measurements and three-axis gyroscope angular velocities (\(\text{rad/s}\)). This array allows us to capture how the device is placed, the user's posture, and the rotational momentum simultaneously.
*   **Block Bootstrap Resampling:** A resampling methodology used here for statistically synthesising mobile signals from historical recordings. Rather than sampling individual, independent data points—which would destroy the temporal dependency and human movement physics—the system uses temporal blocks of accelerometer and gyroscope readings. By stitching these blocks together, we create simulated paths of movement. Blocks are vital as they preserve the physics of the processes (e.g., a step starting when the foot is on the ground) without eliminating cyclical features and autocorrelative dynamics of the timeseries.
*   **Agent Anchor:** The reproducible starting location and activity mode (walking, cycling, or standing) assigned to each simulated agent based on their unique ID. This point acts as the initial coordinate seed from which their full route trajectory is subsequently projected.
*   **Spatial Context:** We map the synthetic agents' coordinates into specific administrative regions (districts) and transit paths. An agent's location is mapped to a district if it falls inside its geographical boundary, or matched to a nearby pedestrian zone or bicycle path if it falls within a 150-metre matching tolerance of the physical infrastructure.
*   **Attraction-Based Routing:** To accurately simulate real-world behaviour, we model how major stations and urban hubs attract citizens. Consequently, \(60\%\) of agents are routed to major Vienna transit stations (*Stephansplatz*, *Karlsplatz*, *Hauptbahnhof*, *Westbahnhof*, *Schottentor*) to mimic commuter travel, while the remaining \(40\%\) disperse to random intersections to simulate ambient city traffic.

> **Note:** Machine Learning classifier definitions will be added here once the model training phase is completed.

---

### 1.2.3. Visualising the Domain Concepts

To make the physical definitions and routing model behaviours clearer, we display the generated signals and spatial outputs obtained from our simulation:

#### 1.2.3.1. Block Bootstrap Resampling
The time-series plot below illustrates how **Block Bootstrap Resampling** preserves the continuous, cyclic patterns of human movement (such as walking and cycling strides) compared to standard random point sampling:

<img src="../data/plots/linear_series.png" alt="Continuous Acceleration Time Series" width="100%"/>
*Figure 1: Block-bootstrapped acceleration time-series (\(\text{m/s}^2\)) across walking, cycling, and standing profiles. Note how the continuous stride frequencies and patterns are cleanly preserved inside each signal block.*

#### 1.2.3.2. Preserving User-to-User Variance
It is vital to ensure that our synthetic agents behave like a realistic population; otherwise, the modelling will be limited by the unique features of the 9 initial respondents. The violin plots below show the resulting physical variation (heterogeneity) in movement metrics across the user pool:

<img src="../data/plots/population_heterogeneity_violins.png" alt="User Population Heterogeneity Violins" width="100%"/>
*Figure 2: Population heterogeneity showing acceleration magnitude distributions across different users. By randomly stratifying our bootstrap seeds across different subjects, our simulated population naturally inherits this physical diversity.*

#### 1.2.3.3. Spatial Context and Commuter Attraction
The scatter map below displays a sample of simulated agent coordinates. It demonstrates how **Agent Anchors** initialize agents, how **Spatial Context** snaps coordinates directly to road and path networks, and how **Attraction-Based Routing** drives agent density toward central transit hubs:

<img src="../data/geospatial_output/telemetry_scatter_map.png" alt="Synthetic Agent Locations by Activity Map" width="100%"/>
*Figure 3: Spatial distribution of simulated agent coordinates in Vienna. Agent coordinates map directly onto the street geometry (spatial context), with walking/cycling routes converging on the central transit stations.*

---

## 1.3. Societal Context and Big Data Characteristics

### 1.3.1. Volume and Population Expansion
Starting with the UCI Heterogeneity Activity Recognition (HHAR) dataset, which contains about 44 million rows of high-frequency kinematic signals from devices, the project processes large datasets. The volume scales significantly because the non-parametric block bootstrap configures these baseline 9 respondents to simulate thousands of new individual agents across the 23 districts of Vienna. To manage, store, and operate this data, we use a distributed Apache Spark environment and work with compressed Parquet storage formats. This prevents memory exhaustion on the nodes and creates a computationally efficient workflow.

### 1.3.2. Velocity and Micro-Batch Stream Processing
As the goal of the work is to simulate the movement of agents across the city, we track data as a continuous influx instead of using static batch processing that evaluates urban states in isolation. Therefore, we use Spark Structured Streaming for the system to ingest data incrementally via a file-source directory. The pipeline processes the accelerometer and gyroscope streams in micro-batches, updating the active state of the simulated transport network in real time. This processing loop forms the time-series foundation needed for further predictive modelling.

### 1.3.3. Variety of Heterogeneous Data Structures
The pipeline is designed for the execution of distributed and multi-layered joins across distinct and varied data formats as unstructured and semi-structured streams are processed:
*   High-frequency, irregularly timed continuous \(X, Y, Z\) kinetic vectors from mobile sensors.
*   Spatial-geospatial vectors: Complex, non-tabular GeoJSON polygons and line strings representing Viennese district boundaries (*Bezirksgrenzen*), pedestrian zones (*Fußgängerzonen*), and bike lane networks (*Radwege*).
*   Structured tabular baselines: Historical municipal CSV and JSON registries mapping population counts and transit metrics across the city.

### 1.3.4. Veracity, Signal De-Noising, and Empirical Grounding
To provide critical accuracy for the simulation and manage sensor noise, we implement algorithmic and empirical validation layers:
*   **Noise Elimination:** Telemetry of such a high frequency is corrupted by arbitrary phone orientations. We clean this data by applying a zero-phase symmetric sliding window (\(2w+1 = 31\text{ samples}\)) to isolate the low-frequency gravitational and directional trend from high-frequency residuals. This lets us formulate a precise signal decomposition.
*   **Empirical Grounding (Municipal Weighting):** To eliminate the risk of an unconstrained simulation, the initial placement and flow of virtual citizens are governed by empirical base rates. Agents are distributed proportionally based on historical passenger metrics (*Fahrgastzahlen der Wiener Linien*) and cycle station data (*Radzählstellenbericht*).

---

# 2. Project Data Sources and Architecture

## 2.1. Empirical and Municipal Data Ingestion Streams

### 2.1.1. Primary Kinematic Telemetry (UCI HHAR Stream)
This dataset provides high-frequency physical measurements captured at a rate of 100 Hz. The pipeline reads the raw CSV rows using Spark with the following layout:
*   **Index (LongType):** Sequential hardware event ordering index.
*   **Arrival_Time (LongType):** Device-level operating system arrival timestamp measured in milliseconds.
*   **Creation_Time (LongType):** Internal hardware sample timestamp measured in nanoseconds.
*   **x, y, z (DoubleType):** Raw continuous sensor readings. For the accelerometer stream, these register directional linear acceleration forces measured in metres per second squared (\(\text{m/s}^2\)). For the gyroscope stream, these register angular rotational velocity values measured in radians per second (\(\text{rad/s}\)).
*   **User (StringType):** Categorical identity key assigned to the nine human baseline respondents ('a' through 'i').
*   **Model / Device (StringType):** Structural tracking attributes recording the specific mobile hardware model and deployment identifier.
*   **gt (StringType):** Ground-truth target kinetic label recording the activity state during data collection ('walk', 'bike', 'sit', 'stand').

| Index | Arrival_Time | Creation_Time | x | y | z | User | Model | Device | gt |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1240101 | 1424696010103 | 14246959901000000 | 0.14582911 | -1.20485912 | 9.71249511 | a | nexus | nexus4 | walk |
| 1240102 | 1424696010113 | 14246960001000000 | 0.22149503 | -0.98402144 | 9.60149202 | a | nexus | nexus4 | walk |

### 2.1.2. Municipal District Boundary Structures (Bezirksgrenzen GeoJSON)
This dataset is sourced directly from the City of Vienna Open Government Data portal (data.gv.at). The vector layer represents the geographical limits of the 23 administrative districts, isolating administrative territories as coordinate polygons:
*   **geometry (Polygon / MultiPolygon):** Absolute EPSG:4326 geographic spatial coordinate arrays mapping out the physical perimeter of each district.
*   **BEZNR (IntegerType):** The formal administrative district number identifier from 1 through 23.
*   **NAME (StringType):** The administrative name designation of the municipal district (e.g., 'Innere Stadt', 'Leopoldstadt', 'Favoriten').
*   **FLAECHE (DoubleType):** The exact calculated spatial footprint surface area of the district polygon measured in square metres (\(\text{m}^2\)).

Literal GeoJSON Attribute Frame:
```json
{
  "type": "Feature",
  "properties": { "BEZNR": 10, "NAME": "Favoriten", "FLAECHE": 54031201.42 },
  "geometry": { "type": "Polygon", "coordinates": [[[16.365, 48.171], [16.372, 48.165], "..."]] }
}
```

2.1.3. Pedestrian Infrastructure Corridors (Fußgängerzonen GeoJSON)
This dataset contains the pedestrian zones across Vienna. The structural attributes within the GeoJSON schema are used to determine safe walking regions for simulating agents:

geometry (Polygon / MultiPolygon): Detailed spatial perimeters defining the explicit walking boundaries.
OBJECTID (IntegerType): Unique municipal registry asset index.
ORTSTEXT (StringType): The street location textual label describing the zone location (e.g., 'Stephansplatz', 'Kärntner Straße').
SHAPE_Area (DoubleType): The geometric area measurement of the pedestrian walkway platform polygon in square metres ((\text{m}^2)). This field is used as the denominator in crowd density tracking calculations.

2.1.3. Pedestrian Infrastructure Corridors (Fußgängerzonen GeoJSON)
This dataset contains the pedestrian zones across Vienna. The structural attributes within the GeoJSON schema are used to determine safe walking regions for simulating agents:

geometry (Polygon / MultiPolygon): Detailed spatial perimeters defining the explicit walking boundaries.
OBJECTID (IntegerType): Unique municipal registry asset index.
ORTSTEXT (StringType): The street location textual label describing the zone location (e.g., 'Stephansplatz', 'Kärntner Straße').
SHAPE_Area (DoubleType): The geometric area measurement of the pedestrian walkway platform polygon in square metres ((\text{m}^2)). This field is used as the denominator in crowd density tracking calculations.

```json
{
  "type": "Feature",
  "properties": { "OBJECTID": 4122, "ORTSTEXT": "Stephansplatz FZO", "SHAPE_Area": 4210.50 },
  "geometry": { "type": "Polygon", "coordinates": "..." }
}
```


2.1.4. Cycling Transport Network Lineations (Radwege GeoJSON)
This layer maps out the entire bicycle infrastructure path grid of Vienna. Unlike the district zones, this layer consists of multi-segmented line paths rather than enclosed area blocks:

geometry (LineString / MultiLineString): Continuous coordinate trajectories mapping out the physical paths of bike corridors.
STRNAM (StringType): The registered name of the underlying urban thoroughfare containing the cycling asset (e.g., 'Lassallestraße', 'Donaukanal Radweg').
RADWEG_TYP (StringType): Categorical class distinguishing the infrastructure layout ('baulich getrennt' for structurally segregated tracks, 'Mehrzweckstreifen' for multi-use painted lanes).
SHAPE_Length (DoubleType): The physical linear length extension of the cycle path element measured in metres ((\text{m})).


Literal GeoJSON Attributes:

```json
{
  "type": "Feature",
  "properties": { "STRNAM": "Lassallestraße", "RADWEG_TYP": "baulich getrennt", "SHAPE_Length": 1240.85 },
  "geometry": { "type": "LineString", "coordinates": [[[16.391, 48.221], [16.402, 48.227]]] }
}
```
